## NeuralHydrology 실습

### 1. Import required libraries and check torch usability in this machine

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from neuralhydrology.evaluation import metrics, get_tester
from neuralhydrology.nh_run import start_run, eval_run
from neuralhydrology.utils.config import Config
from neuralhydrology.utils.configutils import create_config_files as set_config

import torch
# GPU를 사용할 수 없으면 CPU로 학습합니다.
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", device)

### 2. Create Input Dataset for each target station

In [ ]:
# Set Time Periods and Regions
region = "Train_YD_Dam_Downstream"
ar_or_not = "data"
# Select input and target variables for this test case
input = ['date', 'yd_hoflw', '3003430_hrf']
target = ['date', '3004630_hwl']
model = "cudalstm"
basin_dir = str(Path.cwd() / ar_or_not)
basin_txt = f"{basin_dir}/{region}.txt"

In [ ]:
# Set Time Periods and Regions
train_input_csv = f"{region}_hourly.csv"
df = pd.read_csv(f"./{ar_or_not}/training_data/time_series/"+train_input_csv)
df.head(2)

In [ ]:
# Create folders to test, validate and test neuralhydrology model
input_path = f"./{ar_or_not}/training_data/time_series/input/hourly/"+ target[1]
target_path = f"./{ar_or_not}/training_data/time_series/target/hourly/"+ target[1]
# Check whether the specified path exists or not
isExist = os.path.exists(input_path)
if not isExist:
    # Create a new directory because it does not exist
    os.makedirs(input_path)
    print("The new input_path directory is created!")
else: print("The input_path directory is existing!")
# Check whether the specified path exists or not
isExist = os.path.exists(target_path)
if not isExist:
    # Create a new directory because it does not exist
    os.makedirs(target_path)   
    print("The new target_path directory is created!")

In [ ]:
# input and target csv file to test, validate and test neuralhydrology model
input_df = df[input].set_index("date")
target_df = df[target].set_index("date")

# hourlycsv looks for "<basin>.csv" (or "<basin>_input.csv" / "<basin>_target.csv"), so name the files after the basin
input_df.to_csv(f"{input_path}/{region}.csv")
target_df.to_csv(f"{target_path}/{region}.csv")

### 3. Update a Configuration file

In [ ]:
base_yml = f"./{ar_or_not}/Train_Overwrite.yml"

# read a original YAML file
with open(base_yml) as f:
   config = yaml.safe_load(f)

config.update(
     {"experiment_name": target[1]+"_"+model,
                 "run_dir": basin_dir+"/runs/"+ target[1]+"/"+model,
                 "train_basin_file": basin_txt,
                 "validation_basin_file": basin_txt,
                 "test_basin_file": basin_txt,
                 "data_dir": basin_dir+"/training_data",
                 "model": model,
                 "dynamic_inputs": input[1:],
                 "target_variables": target[1:],
                 "clip_targets_to_zero": target[1:],
                 "device": device,
                 }
 )

# save updated YAML file as "Train_Overwrite.yml"
with open(base_yml, "w") as f:
   yaml.safe_dump(config, f, default_flow_style=False)

### 4. Check YAML file for Training Dataset

In [ ]:
config_name = f"{ar_or_not}/Train_Overwrite.yml"

run_config = Config(Path(config_name))
print('model:\t\t', run_config.model)
print('experiment_name:\t', run_config.experiment_name)

print('hidden_size:\t\t', run_config.hidden_size)
print('batch_size:\t\t', run_config.batch_size)
print('seq_length:\t', run_config.seq_length)
print('epochs:\t\t', run_config.epochs)
print('loss:\t\t', run_config.loss)

print('dataset:\t\t', run_config.dataset)
print('forcings:\t\t', run_config.forcings)
print('dynamic_inputs:\t', run_config.dynamic_inputs)
print('target_variables:\t', run_config.target_variables)

### 3. Run NeuralHydrology Model

In [ ]:
%%time
start_run(config_file=Path(config_name))

### 5. Evaluation of NeuralHydrology Model

#### 5.1 Evaluation of Training Data

In [ ]:
import glob
import os
sorted_files = [os.path.join(f"{ar_or_not}/runs/{target[1]}/{run_config.model}/", file) for file in os.listdir(f"{ar_or_not}/runs")]
selected_lists = glob.glob(f'{ar_or_not}/runs/{target[1]}/{run_config.model}/{run_config.experiment_name}*')
output_folder = sorted(selected_lists, key=lambda t: os.stat(t).st_mtime, reverse=True)[0]
output_folder

In [ ]:
run_dir = Path(output_folder)  # you'll find this path in the output of the training above.

# create a tester instance and start evaluation
tester = get_tester(cfg=Config(run_dir / "config.yml"), run_dir=run_dir, period="train", init_model=True)
train_results = tester.evaluate(save_results=False, metrics=run_config.metrics)

train_results.keys()
train_results

In [ ]:
# extract observations and simulations
hourly_qobs = train_results["Train_YD_Dam_Downstream"]["1h"]["xr"][target[1]+"_obs"]
hourly_qsim = train_results["Train_YD_Dam_Downstream"]["1h"]["xr"][target[1]+"_sim"]

fig, ax = plt.subplots(figsize=(16,7))
ax.plot(hourly_qobs["date"], hourly_qobs, label="Observed", c='b')
ax.scatter(hourly_qsim["date"], hourly_qsim, label="Simulated", c='r',s =10)
ax.legend(fontsize=15)
ax.tick_params(axis='both', labelsize=15)
#ax.set_yticklabels(fontsize=15)
ax.set_ylabel("WaterLevel (m)", fontsize=15)
ax.set_title(f"YD-Dam DownStream, {target[1][0:7]} Station, Train period - Hourly NSE {train_results['Train_YD_Dam_Downstream']['1h']['NSE']:.3f}", fontsize=15)
plt.grid()
# Calculate some metrics
values = metrics.calculate_all_metrics(hourly_qobs.isel(time_step=-1), hourly_qsim.isel(time_step=-1))
print("Hourly metrics:")
for key, val in values.items():
    print(f"  {key}: {val:.3f}")

#### 5.2 Evaluation of Validating Data

In [ ]:
run_dir = Path(output_folder) 

# create a tester instance and start evaluation
tester = get_tester(cfg=Config(run_dir / "config.yml"), run_dir=run_dir, period="validation", init_model=True)
val_results = tester.evaluate(save_results=False, metrics=run_config.metrics)

val_results.keys()
val_results

In [ ]:
# extract observations and simulations
hourly_qobs = val_results["Train_YD_Dam_Downstream"]["1h"]["xr"][target[1]+"_obs"]
hourly_qsim = val_results["Train_YD_Dam_Downstream"]["1h"]["xr"][target[1]+"_sim"]

fig, ax = plt.subplots(figsize=(16,7))
ax.plot(hourly_qobs["date"], hourly_qobs, label="Observed", c='b')
ax.scatter(hourly_qsim["date"], hourly_qsim, label="Simulated", c='r',s =10)
ax.legend(fontsize=15)
ax.tick_params(axis='both', labelsize=15)
#ax.set_yticklabels(fontsize=15)
ax.set_ylabel("WaterLevel (m)", fontsize=15)
ax.set_title(f"YD-Dam DownStream, {target[1][0:7]} Station, Validation period - Hourly NSE {val_results['Train_YD_Dam_Downstream']['1h']['NSE']:.3f}", fontsize=15)
plt.grid()
# Calculate some metrics
values = metrics.calculate_all_metrics(hourly_qobs.isel(time_step=-1), hourly_qsim.isel(time_step=-1))
print("Hourly metrics:")
for key, val in values.items():
    print(f"  {key}: {val:.3f}")

#### 5.3 Evaluation of Testing Data

In [ ]:
run_dir = Path(output_folder) 

# create a tester instance and start evaluation
tester = get_tester(cfg=Config(run_dir / "config.yml"), run_dir=run_dir, period="test", init_model=True)
test_results = tester.evaluate(save_results=False, metrics=run_config.metrics)

test_results.keys()
test_results

In [ ]:
# extract observations and simulations
hourly_qobs = test_results["Train_YD_Dam_Downstream"]["1h"]["xr"][target[1]+"_obs"]
hourly_qsim = test_results["Train_YD_Dam_Downstream"]["1h"]["xr"][target[1]+"_sim"]

fig, ax = plt.subplots(figsize=(16,7))
ax.plot(hourly_qobs["date"], hourly_qobs, label="Observed", c='b')
ax.scatter(hourly_qsim["date"], hourly_qsim, label="Simulated", c='r',s =10)
ax.legend(fontsize=15)
ax.tick_params(axis='both', labelsize=15)
#ax.set_yticklabels(fontsize=15)
ax.set_ylabel("WaterLevel (m)", fontsize=15)
ax.set_title(f"YD-Dam DownStream, {target[1][0:7]} Station, Test period - Hourly NSE {test_results['Train_YD_Dam_Downstream']['1h']['NSE']:.3f}", fontsize=15)
plt.grid()
# Calculate some metrics
values = metrics.calculate_all_metrics(hourly_qobs.isel(time_step=-1), hourly_qsim.isel(time_step=-1))
print("Hourly metrics:")
for key, val in values.items():
    print(f"  {key}: {val:.3f}")

In [ ]:
torch.cuda.empty_cache()